# Fine-Tuning with LoRA, QLoRA and PEFT

Fine-tuning a large language model from scratch requires updating billions of parameters, which demands enormous GPU memory and compute. For most use cases, this is neither practical nor necessary.

This notebook walks through three techniques that make fine-tuning accessible on consumer and mid-range hardware:

- **LoRA (Low-Rank Adaptation)** — instead of updating all model weights, LoRA injects small trainable matrices into specific layers. The original weights stay frozen. Only the adapter weights are trained, which is a fraction of the total parameters.
- **QLoRA (Quantized LoRA)** — combines LoRA with 4-bit quantization. The base model is loaded in 4-bit precision (using NF4 format), drastically cutting memory usage. LoRA adapters are trained on top in float16.
- **PEFT (Parameter-Efficient Fine-Tuning)** — the Hugging Face library that implements LoRA, QLoRA, and other adapter methods. It handles attaching, training, saving, and loading adapters cleanly.

**Model:** `facebook/opt-350m` — a lightweight 350M parameter model, ideal for understanding the full fine-tuning pipeline without large VRAM requirements

**Dataset:** `Salesforce/wikitext` (wikitext-2-raw-v1) — Wikipedia articles used to demonstrate domain adaptation on clean, structured text

**Requirements:** CPU or any CUDA-enabled GPU. With 4-bit quantization, OPT-350M runs comfortably on minimal hardware.

## How LoRA Works

LoRA decomposes weight updates into two small matrices instead of updating the full weight matrix. During fine-tuning, the original weights W are **frozen** and only the adapter matrices A and B are trained:

$$h = Wx + \frac{\alpha}{r} \cdot BAx$$

Where:
- **W** (d × d) — the frozen pretrained weight matrix
- **A** (d × r) — down-projection matrix (trainable)
- **B** (r × d) — up-projection matrix (trainable)
- **r** — the rank, much smaller than d (e.g., r=16 vs d=768)
- **α** — scaling factor that controls the magnitude of the adapter update

![LoRA Architecture Diagram](lora_diagram.png)

**Why this works:** LLM weight updates during fine-tuning tend to be low-rank — meaning the important information can be captured by matrices much smaller than the originals. LoRA exploits this by restricting updates to a low-rank subspace.

## How QLoRA Works

QLoRA adds **4-bit quantization** on top of LoRA to further reduce memory:

1. **Base model weights** are quantized to 4-bit NF4 format (frozen, not trained)
2. **LoRA adapter matrices** are kept in float16 (trainable)
3. During the forward pass, 4-bit weights are **dequantized to float16** on the fly for computation
4. Gradients flow only through the LoRA adapters

![QLoRA Architecture Diagram](qlora_diagram.png)

**Memory savings example:**

| Method | Model Size (7B params) | VRAM Required |
|--------|----------------------|---------------|
| Full fine-tuning (fp16) | ~14 GB | ~28 GB+ (with optimizer states) |
| LoRA (fp16 base) | ~14 GB | ~16 GB |
| QLoRA (4-bit base) | ~3.5 GB | ~6 GB |

For our OPT-350M model, QLoRA reduces VRAM from ~700MB to under 200MB.

## Step 1 — Install Dependencies

We need the following libraries:

- `transformers` — loads the base OPT-350M model and tokenizer
- `peft` — provides the LoraConfig and adapter wrapping utilities
- `trl` — provides training utilities for fine-tuning LLMs
- `bitsandbytes` — handles 4-bit quantization (the Q in QLoRA)
- `accelerate` — manages device placement and distributed training
- `datasets` — loads the training data from Hugging Face Hub
- `einops` — a tensor operation library required by some model architectures

In [1]:
!pip install -q -U trl transformers accelerate peft
!pip install -q datasets bitsandbytes einops wandb


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Check Hardware

Before loading any model, it's good practice to check what hardware is available. This tells us whether we have a GPU (and which one), so we know what precision and batch sizes are feasible.

- **GPU available** → we can use CUDA, fp16, and larger batch sizes
- **CPU only** → still works with 4-bit quantization, but training will be slower

In [3]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("No GPU detected — running on CPU")
    print("Training will work but will be slower")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB


## Step 3 — Load the Dataset

We use the WikiText-2 dataset from Salesforce, a collection of Wikipedia articles. Each row has a `text` field containing a passage.

The model will learn the vocabulary, syntax, and writing patterns from Wikipedia-style text. This is a domain adaptation task — we are not teaching the model new facts, but steering it toward a specific style of writing.

You can swap this dataset with any text dataset from Hugging Face Hub. The only requirement is that it has a text column the trainer can read from.

In [4]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

print(f"Dataset size: {len(dataset)} samples")
print("Sample text:")
print(dataset["text"][0][:300])

c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Dataset size: 36718 samples
Sample text:



## Step 4a — Configure QLoRA (4-bit Quantization)

This is the core of QLoRA. `BitsAndBytesConfig` tells the model loader to quantize the weights to 4-bit precision as they are loaded into memory.

Key parameters:

- `load_in_4bit=True` — activates 4-bit quantization
- `bnb_4bit_quant_type="nf4"` — NF4 (NormalFloat4) is a data type designed specifically for normally distributed weights, which LLMs typically have. It gives better accuracy than plain int4.
- `bnb_4bit_compute_dtype=torch.float16` — even though weights are stored in 4-bit, computations happen in float16 for numerical stability

Without quantization, loading OPT-350M in float16 requires ~700MB VRAM. With 4-bit quantization, it fits in under 200MB — making it runnable even on CPU or minimal GPU setups.

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "facebook/opt-350m"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False  # Disable KV cache during training to save memory

print("Model loaded in 4-bit.")

W0608 15:23:13.322000 40168 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

Model loaded in 4-bit.


## Step 4b — Load the Tokenizer

The tokenizer converts raw text into token IDs that the model can process. We load the tokenizer that matches the base model (`facebook/opt-350m`).

One important detail: OPT does not have a dedicated padding token, so we set `pad_token = eos_token`. This tells the tokenizer to use the end-of-sequence token for padding, which is the standard practice for decoder-only models during fine-tuning.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size: {tokenizer.vocab_size}")

Vocabulary size: 50265


## Step 5a — Freeze Base Model Weights and Prepare for Training

Before attaching LoRA adapters, we explicitly freeze all base model parameters and apply some stability optimizations:

1. **Freeze all parameters** — set `requires_grad = False` so the base model weights are not updated during training
2. **Upcast LayerNorm to fp32** — small parameters like layer norms are cast to float32 for numerical stability during mixed-precision training
3. **Enable gradient checkpointing** — trades compute for memory by recomputing activations during the backward pass instead of storing them all
4. **Enable input gradients** — required when using gradient checkpointing with frozen base models

These steps are what PEFT does internally, but doing them explicitly helps you understand what's happening under the hood.

In [7]:
import torch.nn as nn

# Freeze all base model parameters
for param in model.parameters():
    param.requires_grad = False
    if param.ndim == 1:
        # Cast small parameters (e.g., LayerNorm) to fp32 for stability
        param.data = param.data.to(torch.float32)

# Enable gradient checkpointing to reduce memory usage
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Ensure the final output (logits) are in float32 for stable loss computation
class CastOutputToFloat(nn.Sequential):
    def forward(self, x):
        return super().forward(x).to(torch.float32)

model.lm_head = CastOutputToFloat(model.lm_head)

print("Base model frozen and prepared for LoRA training.")

Base model frozen and prepared for LoRA training.


## Step 5b — Understanding Trainable Parameters

Before and after attaching LoRA adapters, it's useful to see exactly how many parameters are trainable. This helper function counts them and shows the percentage — which is the core insight of parameter-efficient fine-tuning.

PEFT provides `model.print_trainable_parameters()`, but writing it manually helps understand the math:

```
trainable% = (trainable_params / all_params) × 100
```

For LoRA with rank 64 on OPT-350M, expect ~1.9% trainable parameters.

In [8]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    Useful for verifying that only adapter weights are being trained.
    """
    trainable_params = 0
    all_params = 0
    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params:,} || "
        f"all params: {all_params:,} || "
        f"trainable%: {100 * trainable_params / all_params:.4f}"
    )

# Before LoRA — should show 0% trainable since we froze everything
print("Before LoRA:")
print_trainable_parameters(model)

Before LoRA:
trainable params: 0 || all params: 179,677,184 || trainable%: 0.0000


## Step 6 — Configure LoRA Adapters (PEFT)

LoRA works by decomposing weight updates into two small matrices: a down-projection matrix A and an up-projection matrix B. Instead of updating the full weight matrix W, it learns the update as `W + alpha * (B @ A)` where B and A are much smaller.

Key parameters:

- `r=64` — the rank of the adapter matrices. Higher rank = more capacity to learn = more parameters. Common values are 8, 16, 32, 64. Start with 16 and increase if the model underfits.
- `lora_alpha=16` — a scaling factor applied to the LoRA update. The effective learning rate of the adapter scales as `alpha / r`. Keeping alpha fixed and increasing r reduces the effective step size.
- `lora_dropout=0.1` — dropout applied to adapter layers to reduce overfitting
- `bias="none"` — do not train bias terms; keeps the adapter lightweight
- `task_type="CAUSAL_LM"` — tells PEFT this is a causal language model task

In [9]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the base model with LoRA adapters
peft_model = get_peft_model(model, peft_config)

# Show trainable params — now only LoRA weights are trainable
print("After LoRA:")
print_trainable_parameters(peft_model)

# Also use PEFT's built-in method for comparison
peft_model.print_trainable_parameters()

After LoRA:
trainable params: 6,291,456 || all params: 185,968,640 || trainable%: 3.3831
trainable params: 6,291,456 || all params: 337,487,872 || trainable%: 1.8642


## Step 7 — Configure Training Arguments

These control how the training loop runs.

Key parameters and why they are set this way:

- `per_device_train_batch_size=4` — number of samples processed per GPU per step. Lower values use less memory.
- `gradient_accumulation_steps=4` — simulates a larger effective batch size (4 x 4 = 16) without needing more VRAM. Gradients are accumulated across 4 steps before an optimizer update.
- `optim="paged_adamw_32bit"` — a memory-efficient variant of AdamW that pages optimizer states to CPU RAM when GPU memory is tight. Essential for QLoRA.
- `learning_rate=2e-4` — a standard starting point for LoRA fine-tuning. Lower than full fine-tuning because adapter weights are sensitive.
- `max_grad_norm=0.3` — gradient clipping to prevent exploding gradients
- `warmup_ratio=0.03` — linearly warm up the learning rate for the first 3% of steps
- `lr_scheduler_type="constant"` — keep LR constant after warmup. Cosine decay also works well.

In [10]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    max_steps=100,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    fp16=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Step 8 — Tokenize the Dataset and Set Up the Trainer

Before training, we need to tokenize the raw text into token IDs. The `tokenize_function` applies the tokenizer to each sample, truncating to `max_length=512` tokens.

We then use the standard Hugging Face `Trainer` with a `DataCollatorForLanguageModeling`:

- **`DataCollatorForLanguageModeling(mlm=False)`** — sets up causal language modeling. The labels are automatically shifted so the model learns to predict the next token at each position.
- **`Trainer`** — the standard Hugging Face training loop. It handles batching, gradient accumulation, logging, checkpointing, and device placement.

`max_length=512` sets the maximum number of tokens per training sample. Longer sequences use more memory quadratically (due to attention). 512 is a safe default; increase to 1024 or 2048 if you have more VRAM.

In [11]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# Data collator for causal language modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Create trainer
trainer = Trainer(
    model=peft_model,
    args=training_arguments,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Trainer ready.")


Trainer ready.


## Step 9 — Train the Model

This runs the training loop. At each step, it:
1. Feeds a batch of text to the model
2. Computes cross-entropy loss (next token prediction)
3. Backpropagates through the adapter layers only (base model weights stay frozen)
4. Updates the LoRA adapter weights

Training loss should decrease steadily. If it plateaus early, try increasing `r` or `learning_rate`. If it diverges, reduce the learning rate.

In [12]:
trainer.train()

Step,Training Loss
10,3.833944
20,3.719442
30,3.717574
40,3.580649
50,3.688522
60,3.713225
70,3.621780
80,3.506991
90,3.530505
100,3.557535


TrainOutput(global_step=100, training_loss=3.6470168685913085, metrics={'train_runtime': 88.0373, 'train_samples_per_second': 18.174, 'train_steps_per_second': 1.136, 'total_flos': 477880692768768.0, 'train_loss': 3.6470168685913085, 'epoch': 0.04357298474945534})

## Step 10 — Save the Adapter Weights

After training, we save only the LoRA adapter weights — not the entire model. The adapter is a small set of matrices (typically a few MB to ~150MB depending on rank and target modules) rather than the full 350M parameter base model.

This is one of the key advantages of PEFT: you can version and distribute adapters independently, and apply them on top of any compatible base model at inference time.

In [13]:
# Handle both single GPU and distributed/parallel training setups
model_to_save = trainer.model.module if hasattr(trainer.model, "module") else trainer.model
model_to_save.save_pretrained("outputs")

print("Adapter weights saved to ./outputs")

Adapter weights saved to ./outputs


## Step 11 — Load the Adapter and Run Inference

To use the fine-tuned model, we load the saved LoRA adapter from the outputs directory using `PeftModel.from_pretrained()`. This attaches the trained adapter weights back onto the base model.

At inference time, the adapter weights are merged with the base model's weights on the fly, so there is no speed penalty compared to the base model.

We test with a Wikipedia-style prompt since the model was fine-tuned on WikiText data.

### Loading from a local directory vs. Hugging Face Hub

The same `PeftModel.from_pretrained()` method works for both:
- **Local:** `PeftModel.from_pretrained(base_model, "./outputs")`
- **Hub:** `PeftModel.from_pretrained(base_model, "username/adapter-repo")`

You can also use `PeftConfig.from_pretrained()` to inspect the adapter configuration before loading — this tells you which base model the adapter was trained on, the rank, alpha, and target modules.

In [14]:
from peft import PeftModel, PeftConfig

# Inspect the adapter config (useful when loading from Hub)
adapter_config = PeftConfig.from_pretrained("outputs")
print(f"Base model: {adapter_config.base_model_name_or_path}")
print(f"LoRA rank: {adapter_config.r}")
print(f"LoRA alpha: {adapter_config.lora_alpha}")
print(f"Task type: {adapter_config.task_type}")

# Load the saved adapter onto the base model
inference_model = PeftModel.from_pretrained(model, "outputs")

print("\nAdapter loaded successfully.")

Base model: facebook/opt-350m
LoRA rank: 64
LoRA alpha: 16
Task type: CAUSAL_LM

Adapter loaded successfully.


c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\tuners\tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [15]:
# Test the fine-tuned model with a Wikipedia-style prompt
text = "The history of artificial intelligence began in"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

inputs = tokenizer(text, return_tensors="pt").to(device)

# Use autocast for mixed-precision inference (matches training precision)
with torch.cuda.amp.autocast() if torch.cuda.is_available() else torch.no_grad():
    outputs = inference_model.generate(**inputs, max_new_tokens=100)

print("Prompt:", text)
print("\nGenerated:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

C:\Users\Radhakrishna\AppData\Local\Temp\ipykernel_40168\1837285674.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast() if torch.cuda.is_available() else torch.no_grad():
c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Prompt: The history of artificial intelligence began in

Generated:
The history of artificial intelligence began in the early 1990s with the development of the first artificial intelligence (AI) system, called the "AI Brain" by the University of California at Berkeley. The first AI system was developed by the University of California at Berkeley in the late 1990s and was used to create the first artificial intelligence (AI) system for the University of California at Berkeley. The first AI system was developed by the University of California at Berkeley in the late 1990s and was used to create the first artificial intelligence (AI)
